### Google Drive Mount
This cell mounts your Google Drive to the Colab environment so that the dataset and model weights stored in Drive can be accessed.

In [1]:
from google.colab import drive
drive.mount('/content/drive')


ValueError: mount failed

Imports, Model Loading, and Target Layer Setup

### Environment Setup & Directory Configuration
This cell imports essential libraries (TensorFlow, OpenCV, Pandas, etc.) and configures the correct paths for the dataset images and the ResNet50 model weights.

In [ ]:
import os
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.applications.resnet50 import preprocess_input
from tensorflow.keras.preprocessing.image import load_img, img_to_array

if 'folder_path' not in locals():
    folder_path = "/content/drive/MyDrive/InnoTech_Dataset"

images_dir = os.path.join(folder_path, 'images_224')
if not os.path.exists(images_dir):
    images_dir = os.path.join(folder_path, 'images')

model_path = os.path.join(folder_path, 'ResNet50_merged_fixed.keras')

Grad-CAM Core Computation

### Model Loading and Feature Extractor Setup
This cell loads the pre-trained Keras model and extracts the target convolutional layer (`conv4_block6_out`) alongside the final network outputs to build the Grad-CAM computation model.

In [ ]:
model = keras.models.load_model(model_path)

resnet_backbone = model.layers[0] if isinstance(model.layers[0], keras.Model) else model
head_layers = model.layers[1:] if resnet_backbone != model else []

TARGET_LAYER_NAME = 'conv4_block6_out'
target_conv_layer = resnet_backbone.get_layer(TARGET_LAYER_NAME)

base_grad_model = keras.Model(
    inputs=resnet_backbone.inputs,
    outputs=[target_conv_layer.output, resnet_backbone.output]
)

print(f"Extractor ready using layer: {TARGET_LAYER_NAME}")

Visualization & Overlay Function

### Grad-CAM Computation & Visualization Functions
This cell defines the core helper functions: `compute_gradcam` calculates gradients and activation maps, while `run_and_display_gradcam` applies the heatmap onto the original image and visualizes the prediction results.

In [ ]:
def compute_gradcam(img_tensor):
    with tf.GradientTape() as tape:
        conv_outputs, backbone_output = base_grad_model(img_tensor)
        tape.watch(conv_outputs)

        x = backbone_output
        for layer in head_layers:
            x = layer(x)

        score = x[0][0]

    grads = tape.gradient(score, conv_outputs)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))

    conv_outputs = conv_outputs[0]
    heatmap = conv_outputs @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)

    heatmap = tf.maximum(heatmap, 0)
    max_val = tf.reduce_max(heatmap)
    if max_val != 0:
        heatmap /= max_val

    return heatmap.numpy(), float(score.numpy())


def run_and_display_gradcam(image_path, save_result=True):
    raw_img = load_img(image_path, target_size=(224, 224))
    raw_img_array = img_to_array(raw_img)

    img_for_model = np.expand_dims(raw_img_array.copy(), axis=0)
    img_preprocessed = preprocess_input(img_for_model)

    heatmap, score = compute_gradcam(img_preprocessed)

    pred_label = "PNEUMONIA" if score >= 0.5 else "NORMAL"
    confidence = score if score >= 0.5 else (1.0 - score)

    heatmap_resized = cv2.resize(heatmap, (224, 224))
    heatmap_colored = cv2.applyColorMap(np.uint8(255 * heatmap_resized), cv2.COLORMAP_JET)
    heatmap_colored = cv2.cvtColor(heatmap_colored, cv2.COLOR_BGR2RGB)

    overlay = cv2.addWeighted(np.uint8(raw_img_array), 0.65, heatmap_colored, 0.35, 0)

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    filename = os.path.basename(image_path)

    axes[0].imshow(raw_img_array.astype("uint8"))
    axes[0].set_title(f"Original Image\n{filename}")
    axes[0].axis("off")

    axes[1].imshow(heatmap_resized, cmap="jet")
    axes[1].set_title(f"Grad-CAM ({TARGET_LAYER_NAME})")
    axes[1].axis("off")

    axes[2].imshow(overlay)
    axes[2].set_title(f"Pred: {pred_label} ({confidence*100:.1f}%)")
    axes[2].axis("off")

    plt.tight_layout()

    if save_result:
        out_dir = os.path.join(folder_path, "gradcam_output")
        os.makedirs(out_dir, exist_ok=True)
        plt.savefig(os.path.join(out_dir, f"gradcam_{filename}"), dpi=150, bbox_inches='tight')

    plt.show()

### High-Confidence Test Sample Evaluation
This cell filters the test dataset to locate, execute, and save Grad-CAM visualizations for the first 5 samples predicted with high confidence (scores >= 80% or <= 20%).

In [ ]:
csv_path = os.path.join(folder_path, 'cnn_data.csv')

if os.path.exists(csv_path):
    df = pd.read_csv(csv_path)
    test_df = df[df['split'] == 'test'].reset_index(drop=True)

    found_count = 0
    for idx, row in test_df.iterrows():
        img_path = os.path.join(images_dir, row['filename'])
        if os.path.exists(img_path):
            raw_img = load_img(img_path, target_size=(224, 224))
            img_array = img_to_array(raw_img)
            img_tensor = preprocess_input(np.expand_dims(img_array.copy(), axis=0))

            _, score = compute_gradcam(img_tensor)

            if score >= 0.80 or score <= 0.20:
                print(f"High-Confidence Sample: {row['filename']} | Score: {score:.3f}")
                run_and_display_gradcam(img_path, save_result=True)
                found_count += 1

            if found_count >= 5:
                break

### Random Test Sample Visualization
This cell randomly selects 5 images from the test set, processes them through the Grad-CAM pipeline, displays the visual overlays, and saves the final outputs to your Google Drive.

In [ ]:
csv_path = os.path.join(folder_path, 'cnn_data.csv')

if os.path.exists(csv_path):
    df = pd.read_csv(csv_path)
    test_df = df[df['split'] == 'test'].reset_index(drop=True)

    # Select 5 random images from the test set
    random_samples = test_df.sample(n=5)

    for idx, row in random_samples.iterrows():
        img_path = os.path.join(images_dir, row['filename'])
        if os.path.exists(img_path):
            print(f"Processing Random Image: {row['filename']} | True Label: {row['label']}")
            run_and_display_gradcam(img_path, save_result=True)
        else:
            print(f"File not found: {img_path}")